# Simplificare documente juridice românești în limbaj clar (plain language)

An OpenAI Agents SDK pipeline that turns Romanian legal PDFs into plain-language versions,
measures the readability change with ReaderBench-style lexical indices, and writes a
packaged PDF for each document.

## Flow
1. **Load** legal PDFs from `data/`.
2. **Agent 1 — Simplificator**: rewrites the text into plain Romanian, following `criteria.md`.
3. **Readability**: compute lexical indices (ICTAI 2016, Sec. III.B.1) on the original and the
   simplified text with spaCy, then compare.
4. **Agent 2 — Redactor rezumat**: extracts the key points.
5. **Package**: a deterministic writer builds `simplified_docs/<name>_simplified.pdf` containing the
   summary, the readability comparison, and the plain-language version.

## Prerequisites
```bash
pip install openai-agents pdfplumber spacy fpdf2 python-dotenv matplotlib
python -m spacy download ro_core_news_lg
```
- Put an `OPENAI_API_KEY` in a `.env` file next to the notebook (or in your environment).
- Put input PDFs in `data/` and keep `criteria.md` next to the notebook.

## Design notes
- **ReaderBench, lexical subset.** We compute the Sec. III.B.1 lexical indices directly with the
  Romanian spaCy model rather than the `rbpy-rb` package, which pins old dependencies. The
  cohesion/semantic indices (Sec. III.B.2, LSA/LDA) are intentionally out of scope; the scoring
  function is easy to extend if you later add trained Romanian semantic models.
- **spaCy lemmas as stem proxy.** The paper's two stem-distance indices are collapsed into one
  `word–lemma` edit distance, because spaCy exposes lemmas, not Romanian stems.
- **The agent writes content; a function renders the PDF.** LLMs don't render PDFs reliably, so
  Agent 2 produces the summary text and a deterministic writer assembles the file (with a Unicode
  font so `ă â î ș ț` render correctly).
- **Legal fidelity.** Simplification can shift legal meaning, so each PDF carries a short
  "orientational, not legal advice" disclaimer.

In [ ]:
import os
import sys
import math
from pathlib import Path

from dotenv import load_dotenv
from agents import Agent, Runner, trace, set_tracing_export_api_key

DEFAULT_MODEL = 'gpt-5.4-mini'

## Setup: environment and folders

In [ ]:
load_dotenv(Path.cwd() / '.env')
load_dotenv(Path.cwd().parent / '.env')
assert os.getenv('OPENAI_API_KEY'), 'Set OPENAI_API_KEY in .env or your environment before running this notebook.'
set_tracing_export_api_key(os.environ['OPENAI_API_KEY'])

# Resolve folders whether the notebook runs from the repo root or a notebooks/ subfolder.
candidate_data_dirs = [Path.cwd() / 'data', Path.cwd().parent / 'data']
DATA_DIR = next((p for p in candidate_data_dirs if p.exists()), candidate_data_dirs[0])
OUTPUT_DIR = DATA_DIR.parent / 'simplified_docs'
OUTPUT_DIR.mkdir(exist_ok=True)

candidate_criteria = [Path.cwd() / 'criteria.md', Path.cwd().parent / 'criteria.md']
CRITERIA_PATH = next((p for p in candidate_criteria if p.exists()), candidate_criteria[0])
assert CRITERIA_PATH.exists(), f'Missing criteria file. Place criteria.md next to the notebook (looked for {CRITERIA_PATH}).'

print('Data dir:  ', DATA_DIR)
print('Output dir:', OUTPUT_DIR)
print('Criteria:  ', CRITERIA_PATH)

## Step 1: Load legal PDFs from `data/`

In [ ]:
import pdfplumber


def extract_pdf_text(path: Path) -> str:
    parts = []
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            parts.append(page.extract_text() or '')
    return '\n'.join(parts).strip()


def load_documents() -> list[dict]:
    docs = []
    for path in sorted(DATA_DIR.glob('*.pdf')):
        text = extract_pdf_text(path)
        if text:
            docs.append({'filename': path.name, 'stem': path.stem, 'text': text})
        else:
            print(f'Skipped (no extractable text): {path.name}')
    if not docs:
        raise FileNotFoundError(f'No readable .pdf files found in {DATA_DIR}. Add PDFs and rerun.')
    return docs


DOCUMENTS = load_documents()
print(f'Loaded {len(DOCUMENTS)} document(s):', [d['filename'] for d in DOCUMENTS])

## Step 2: Readability — lexical indices (ICTAI 2016, Sec. III.B.1)

All five indices are content-word based where the paper specifies it. For every index here a
**lower value means simpler / more readable** text, which is what plain-language editing should do.

| Index | What it captures |
|---|---|
| `avg_word_length_chars` | shorter words |
| `avg_words_per_sentence` | shorter sentences |
| `unique_content_words_per_sentence` | less lexical load per sentence |
| `word_entropy` | lower lexical diversity / more consistent wording |
| `avg_word_lemma_distance` | lighter morphological inflection (stem-distance proxy) |

In [ ]:
import spacy

NLP = spacy.load('ro_core_news_lg')
CONTENT_POS = {'NOUN', 'PROPN', 'VERB', 'ADJ', 'ADV'}


def _edit_distance(a: str, b: str) -> int:
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]


def lexical_indices(text: str) -> dict:
    doc = NLP(text)
    sentences = [s for s in doc.sents if any(t.is_alpha for t in s)]
    words = [t for t in doc if t.is_alpha]
    content = [t for t in words if not t.is_stop and t.pos_ in CONTENT_POS]

    n_sent = max(len(sentences), 1)
    n_words = max(len(words), 1)
    n_content = max(len(content), 1)

    avg_word_length = sum(len(t.text) for t in words) / n_words
    avg_words_per_sentence = len(words) / n_sent

    unique_per_sentence = sum(
        len({t.lemma_.lower() for t in s
             if t.is_alpha and not t.is_stop and t.pos_ in CONTENT_POS})
        for s in sentences
    ) / n_sent

    counts = {}
    for t in content:
        lem = t.lemma_.lower()
        counts[lem] = counts.get(lem, 0) + 1
    total = sum(counts.values()) or 1
    word_entropy = -sum((c / total) * math.log2(c / total) for c in counts.values())

    word_lemma_distance = sum(
        _edit_distance(t.text.lower(), t.lemma_.lower()) for t in content
    ) / n_content

    return {
        'avg_word_length_chars': round(avg_word_length, 3),
        'avg_words_per_sentence': round(avg_words_per_sentence, 3),
        'unique_content_words_per_sentence': round(unique_per_sentence, 3),
        'word_entropy': round(word_entropy, 3),
        'avg_word_lemma_distance': round(word_lemma_distance, 3),
    }

In [ ]:
INDEX_LABELS = {
    'avg_word_length_chars': 'Lungime medie cuvant (caractere)',
    'avg_words_per_sentence': 'Lungime medie propozitie (cuvinte)',
    'unique_content_words_per_sentence': 'Cuvinte de continut unice / propozitie',
    'word_entropy': 'Entropia cuvintelor (diversitate lexicala)',
    'avg_word_lemma_distance': 'Distanta medie cuvant-lema (flexiune)',
}
# Lower is simpler for every index above.


def compare_indices(original: dict, simplified: dict) -> list[dict]:
    rows = []
    for key, label in INDEX_LABELS.items():
        o, s = original[key], simplified[key]
        if s < o:
            evaluare = 'mai simplu'
        elif s > o:
            evaluare = 'mai complex'
        else:
            evaluare = 'neschimbat'
        rows.append({'index': label, 'original': o, 'simplified': s,
                     'delta': round(s - o, 3), 'evaluare': evaluare})
    return rows

## Step 3: Agent 1 — Simplificator (loads `criteria.md`)

In [ ]:
CRITERIA = CRITERIA_PATH.read_text(encoding='utf-8')

simplifier_agent = Agent(
    name='Simplificator limbaj juridic',
    instructions=(
        'Esti un expert in limbaj administrativ clar (plain language) pentru limba romana. '
        'Rescrii textul juridic primit intr-o versiune in limbaj simplu, pastrand sensul juridic '
        'exact: toate obligatiile, drepturile, termenele si conditiile. Nu adauga si nu elimina '
        'informatie de fond.\n\nAplica urmatoarele criterii:\n\n' + CRITERIA +
        '\n\nRaspunde DOAR cu textul simplificat in limba romana, fara preambul si fara marcaje Markdown.'
    ),
    model=DEFAULT_MODEL,
)


async def simplify(text: str) -> str:
    with trace('Simplificare limbaj juridic'):
        result = await Runner.run(simplifier_agent, text)
    return result.final_output.strip()

## Step 4: Agent 2 — Redactor rezumat (key points)

In [ ]:
report_writer_agent = Agent(
    name='Redactor rezumat',
    instructions=(
        'Primesti un document juridic in limba romana. Extrage cele mai importante puncte pentru '
        'cititor: partile implicate, obligatiile principale, drepturile, termenele si eventualele '
        'sanctiuni. Scrie 4-7 puncte concise, in limbaj simplu, fiecare pe cate o linie care incepe '
        'cu "- ". Raspunde DOAR cu lista de puncte, fara titlu si fara alt text.'
    ),
    model=DEFAULT_MODEL,
)


async def summarize(text: str) -> str:
    with trace('Rezumat puncte cheie'):
        result = await Runner.run(report_writer_agent, text)
    return result.final_output.strip()

## Step 5: Package the PDF (deterministic, Unicode font)

The writer registers a DejaVu font so Romanian diacritics render. It searches common font
locations, including the copy bundled with matplotlib.

In [ ]:
from fpdf import FPDF


def _find_font(filename: str):
    search = []
    try:
        import matplotlib
        search.append(Path(matplotlib.get_data_path()) / 'fonts' / 'ttf')
    except Exception:
        pass
    search += [Path('/usr/share/fonts/truetype/dejavu'),
               Path('/Library/Fonts'), Path('C:/Windows/Fonts')]
    for d in search:
        p = d / filename
        if p.exists():
            return str(p)
    return None


REGULAR_TTF = _find_font('DejaVuSans.ttf')
BOLD_TTF = _find_font('DejaVuSans-Bold.ttf') or REGULAR_TTF
assert REGULAR_TTF, ('DejaVuSans.ttf not found. Install matplotlib (it bundles the font) or set '
                     'REGULAR_TTF / BOLD_TTF to a Unicode .ttf so Romanian diacritics render.')

DISCLAIMER = ('Acest document este o versiune orientativa in limbaj simplu, generata automat. '
              'Nu constituie consultanta juridica si nu inlocuieste textul oficial.')


def build_pdf(out_path: Path, title: str, summary: str, rows: list[dict], simplified_text: str) -> Path:
    pdf = FPDF()
    pdf.set_auto_page_break(auto=True, margin=15)
    pdf.add_font('DejaVu', '', REGULAR_TTF)
    pdf.add_font('DejaVu', 'B', BOLD_TTF)
    pdf.add_page()

    pdf.set_font('DejaVu', 'B', 15)
    pdf.multi_cell(0, 9, title)
    pdf.ln(2)

    pdf.set_font('DejaVu', '', 9)
    pdf.set_text_color(90, 90, 90)
    pdf.multi_cell(0, 5, DISCLAIMER)
    pdf.set_text_color(0, 0, 0)
    pdf.ln(4)

    pdf.set_font('DejaVu', 'B', 12)
    pdf.multi_cell(0, 8, 'Rezumat - puncte importante')
    pdf.set_font('DejaVu', '', 11)
    pdf.multi_cell(0, 6, summary)
    pdf.ln(4)

    pdf.set_font('DejaVu', 'B', 12)
    pdf.multi_cell(0, 8, 'Comparatie lizibilitate (indici lexicali ReaderBench)')
    pdf.ln(1)

    headers = ['Indice', 'Original', 'Simplificat', 'Delta', 'Evaluare']
    widths = [72, 24, 26, 20, 30]
    pdf.set_font('DejaVu', 'B', 9)
    for h, w in zip(headers, widths):
        pdf.cell(w, 7, h, border=1)
    pdf.ln(7)

    pdf.set_font('DejaVu', '', 9)
    for row in rows:
        values = [row['index'], f"{row['original']}", f"{row['simplified']}",
                  f"{row['delta']:+.3f}", row['evaluare']]
        x0, y0 = pdf.get_x(), pdf.get_y()
        pdf.multi_cell(widths[0], 7, values[0], border=1)
        y1 = pdf.get_y()
        h = y1 - y0
        pdf.set_xy(x0 + widths[0], y0)
        for v, w in zip(values[1:], widths[1:]):
            pdf.cell(w, h, v, border=1)
        pdf.set_xy(x0, y1)

    pdf.ln(2)
    pdf.set_font('DejaVu', '', 8)
    pdf.set_text_color(90, 90, 90)
    pdf.multi_cell(0, 4, 'Nota: valori mai mici indica, in general, un text mai simplu si mai usor de citit.')
    pdf.set_text_color(0, 0, 0)
    pdf.ln(4)

    pdf.set_font('DejaVu', 'B', 12)
    pdf.multi_cell(0, 8, 'Versiune in limbaj simplu')
    pdf.set_font('DejaVu', '', 11)
    pdf.multi_cell(0, 6, simplified_text)

    pdf.output(str(out_path))
    return out_path

## Step 6: Run the pipeline end to end

For each document: simplify -> summarize -> score original vs. simplified -> write the PDF.

In [ ]:
async def process_document(doc: dict) -> Path:
    print(f"\n=== {doc['filename']} ===")
    original = doc['text']

    simplified = await simplify(original)
    summary = await summarize(original)

    rows = compare_indices(lexical_indices(original), lexical_indices(simplified))
    print('Readability comparison (original -> simplified):')
    for r in rows:
        print(f"  {r['index']:<40} {r['original']:>8} -> {r['simplified']:>8}  ({r['evaluare']})")

    out_path = OUTPUT_DIR / f"{doc['stem']}_simplified.pdf"
    title = f"{doc['stem'].replace('_', ' ')} - versiune simplificata"
    with trace(f"Generare PDF {doc['filename']}"):
        build_pdf(out_path, title, summary, rows, simplified)
    print('Saved:', out_path)
    return out_path


results = []
for _doc in DOCUMENTS:
    results.append(await process_document(_doc))

print('\nDone. Files written:')
for p in results:
    print(' -', p)